In [1]:
%pip install reportlab pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from io import BytesIO
from datetime import datetime

from PIL import Image as PILImage

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    Image as PDFImage
)

# Main project folder
PROJECT_DIR = Path(
    r"C:\Users\adhil\OneDrive\Desktop\dfu\DFU_Classification_Project"
)

# Folder for patient PDFs
OUTPUT_DIR = PROJECT_DIR / "outputs" / "patient_reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Patient report folder:", OUTPUT_DIR)
print("Folder ready:", OUTPUT_DIR.exists())

Patient report folder: C:\Users\adhil\OneDrive\Desktop\dfu\DFU_Classification_Project\outputs\patient_reports
Folder ready: True


In [3]:

def generate_patient_report(
    patient_details,
    foot_image,
    predicted_grade,
    confidence,
    gradcam_image=None
):
    """Create a patient-specific DFU assessment PDF."""

    pdf_buffer = BytesIO()

    document = SimpleDocTemplate(
        pdf_buffer,
        pagesize=A4,
        rightMargin=1.8 * cm,
        leftMargin=1.8 * cm,
        topMargin=1.8 * cm,
        bottomMargin=1.8 * cm
    )

    styles = getSampleStyleSheet()

    styles.add(ParagraphStyle(
        name="PatientTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        textColor=colors.HexColor("#174A7E"),
        fontSize=20
    ))

    content = []

    content.append(Paragraph(
        "DFU Patient Assessment Report",
        styles["PatientTitle"]
    ))
    content.append(Paragraph(
        "AI-assisted experimental assessment",
        styles["Normal"]
    ))
    content.append(Spacer(1, 20))

    # Patient details
    content.append(Paragraph(
        "1. Patient Details", styles["Heading2"]
    ))

    patient_rows = [
        ["Field", "Details"],
        ["Patient ID", str(patient_details.get("patient_id", "Not provided"))],
        ["Name", str(patient_details.get("name", "Not provided"))],
        ["Age", str(patient_details.get("age", "Not provided"))],
        ["Gender", str(patient_details.get("gender", "Not provided"))],
        ["Assessment Date", datetime.now().strftime("%d-%m-%Y %H:%M")]
    ]

    patient_table = Table(
        patient_rows,
        colWidths=[5 * cm, 11 * cm]
    )
    patient_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#174A7E")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("PADDING", (0, 0), (-1, -1), 8)
    ]))

    content.append(patient_table)
    content.append(Spacer(1, 20))

    # Prediction results
    content.append(Paragraph(
        "2. Prediction Result", styles["Heading2"]
    ))

    confidence = float(confidence)
    if confidence > 1:
        confidence = confidence / 100

    result_table = Table([
        ["Predicted Grade", str(predicted_grade)],
        ["Model Confidence", f"{confidence:.2%}"]
    ], colWidths=[5 * cm, 11 * cm])

    result_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#E8F1F8")),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("PADDING", (0, 0), (-1, -1), 8)
    ]))

    content.append(result_table)
    content.append(Spacer(1, 20))

    # Helper to add an image to the PDF
    def add_image(image_source):
        if image_source is None:
            return

        if isinstance(image_source, PILImage.Image):
            img = image_source.convert("RGB")
        else:
            if hasattr(image_source, "seek"):
                image_source.seek(0)
            img = PILImage.open(image_source).convert("RGB")

        width, height = img.size
        scale = min(
            (15 * cm) / width,
            (9 * cm) / height
        )

        image_buffer = BytesIO()
        img.save(image_buffer, format="PNG")
        image_buffer.seek(0)

        content.append(PDFImage(
            image_buffer,
            width=width * scale,
            height=height * scale
        ))
        content.append(Spacer(1, 15))

    # Uploaded foot image
    content.append(Paragraph(
        "3. Uploaded Foot Image", styles["Heading2"]
    ))
    add_image(foot_image)

    # Optional Grad-CAM image
    if gradcam_image is not None:
        content.append(Paragraph(
            "4. Grad-CAM Visualization", styles["Heading2"]
        ))
        content.append(Paragraph(
            "This visualization highlights image regions that influenced "
            "the model prediction. It does not confirm a medical condition.",
            styles["Normal"]
        ))
        content.append(Spacer(1, 10))
        add_image(gradcam_image)

    # Disclaimer
    content.append(Paragraph(
        "Important Disclaimer", styles["Heading2"]
    ))
    content.append(Paragraph(
        "This report contains an experimental AI prediction, not a medical "
        "diagnosis. The model achieved approximately 65.96% test accuracy "
        "on the current 141-image test dataset. Predictions may be incorrect "
        "and must not replace assessment by a qualified healthcare professional.",
        styles["Normal"]
    ))

    document.build(content)

    # Return PDF bytes for a future Streamlit download button
    return pdf_buffer.getvalue()

print("PDF-generation function defined successfully!")


PDF-generation function defined successfully!


In [4]:
from pathlib import Path

TEST_IMAGE_FOLDER = (
    PROJECT_DIR / "data" / "dfu 4 class" / "test" / "Grade 1"
)

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

test_images = [
    p for p in TEST_IMAGE_FOLDER.iterdir()
    if p.is_file() and p.suffix.lower() in image_extensions
]

if not test_images:
    raise FileNotFoundError(
        f"No test images found in: {TEST_IMAGE_FOLDER}"
    )

sample_image = test_images[0]

print("Sample image:", sample_image)
print("Image exists:", sample_image.exists())

Sample image: C:\Users\adhil\OneDrive\Desktop\dfu\DFU_Classification_Project\data\dfu 4 class\test\Grade 1\112_jpg.rf.472615b206cdec5593b07c5716920e47.jpg
Image exists: True


In [5]:

# Fictional patient details for testing
patient_details = {
    "patient_id": "DEMO-001",
    "name": "Demo Patient",
    "age": "45",
    "gender": "Not specified"
}

# Placeholder values ONLY for testing the PDF layout
predicted_grade = "Grade 1"
confidence = 0.82

# Generate the PDF
pdf_bytes = generate_patient_report(
    patient_details=patient_details,
    foot_image=sample_image,
    predicted_grade=predicted_grade,
    confidence=confidence,
    gradcam_image=None
)

# Save the PDF
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pdf_path = OUTPUT_DIR / "DEMO_patient_assessment_report.pdf"
pdf_path.write_bytes(pdf_bytes)

print("PDF created successfully!")
print("Saved to:", pdf_path)
print("File size:", pdf_path.stat().st_size, "bytes")


PDF created successfully!
Saved to: C:\Users\adhil\OneDrive\Desktop\dfu\DFU_Classification_Project\outputs\patient_reports\DEMO_patient_assessment_report.pdf
File size: 165932 bytes
